## uncomment this and run in the terminal if you need a venv

<!-- create the venv -->
<!-- Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process -->
<!-- py -m venv .venv -->
<!-- .venv\scripts\activate -->
<!-- py -m pip install --upgrade pip -->
<!-- py -m pip install Faker -->
<!-- py -m pip install datetime -->
<!-- py -m pip install pandas -->
<!-- py -m pip install numpy -->
<!-- py -m pip install duckdb==0.7.0 -->

<!-- pip list -->

## Import some packages

In [1]:
!pip install pandas
!pip install names
!pip install faker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.1/789.1 kB 6.1 MB/s  0:00:000:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for names: filename=names-0.3.0-py3-none-any.whl size=803749 sha256=cc1cce7db3f620898b0a28971a7bd3cc581ed3f6f83e22c24967284b8b047957
  Stored in directory: /home/jovyan/.cache/pip/wheels/bd/d4/8f/03181acc8ebae4403c4a16423e9f68a0b0e65496d607d43462
Successfully built names
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.6 MB/s  0:00:00m eta 0:00:01


In [2]:
import pandas as pd
# pd.set_option('display.max_rows', 100)
import random
import names
import numpy as np
import datetime
from faker import Faker

# fake date source: https://stackoverflow.com/questions/553303/generate-a-random-date-between-two-other-dates

# uncomment and use your favorite seed if you want it to be reproducible
# random.seed(94025)
# Faker.seed(94025)
fake = Faker()

In [3]:
def random_country():
    countries = ["Denmark", "Germany", "Sweden", "Norway", "USA", "Canada", "UK", "Netherlands", "Switzerland", "Austria", "France", "Spain"]
    probabilities = [0.15, 0.31, 0.06, 0.05, 0.3, 0.1, 0.1, 0.05, 0.05, 0.1, 0.12, 0.05]
    
    return random.choices(countries, weights=probabilities)[0]

## Make sales table

In [4]:
customer_count = 10000
number_of_purchases = 10000

customer_names = []
purchase_dates = []
purchasing_customers = []
purchased_items = []

# start by faking some names
for i in range(0,customer_count):
    customer_names.append(fake.name())
customer_names = np.unique(customer_names)
num_of_customers = len(customer_names)

# now let's fake some purchases, then assign a random customer name and item
for i in range(0,number_of_purchases):
    purchase_dates.append(fake.date_between(start_date='-100d', end_date='today').isoformat())
    purchasing_customers.append(random.randint(1,num_of_customers))
    purchased_items.append(random.sample(['Dinner table','Sofa','Bed'], 1)[0])

q2_table_purchases = pd.DataFrame({"date":purchase_dates,
                                    "customer_id":purchasing_customers,
                                    "item":purchased_items})

## Make customers table

In [5]:
countries = []
customer_emails = []

for i in range(0,num_of_customers):
    countries.append(random_country())
    customer_emails.append(fake.ascii_email())

customers_table = pd.DataFrame({"id":list(range(1,num_of_customers+1)),
                                    "customer":customer_names,
                                    "email":customer_emails,
                                    "country":countries})

## Make marketing table

In [6]:
non_customer_count = 100
total_number_of_touches = 100000 # trying to keep this accessible so someone could replicate in excel if needed
channels = ['Direct','Email','Organic','Referral','Unknown'] # ???

touched_customer_emails = []
non_customer_emails = []
touch_dates = []
touch_channels = []

# just to keep it fun, there should be some marketing touches that didn't convert/end up in the purchases table
for i in range(0,non_customer_count):
    non_customer_emails.append(fake.ascii_email())

# we already removed duplicates from `customers` but
#   there could be duplicates in non_customers
#   there could also be overlap between the two sets
# so just to be safe:
all_customer_emails = np.unique(list(set(non_customer_emails).union(set(customer_emails))))


for i in range(0,total_number_of_touches):
    touch_dates.append(fake.date_between(start_date='-180d', end_date='today').isoformat())
    touched_customer_emails.append(random.sample(sorted(all_customer_emails), 1)[0])
    touch_channels.append(random.sample(channels, 1)[0])

# there's a chance we have no marketing entry before the purchase date
# not worth fixing
# if someone checks/notices/cares about that, they've probably already passed

q3_table_marketing = pd.DataFrame({"date":touch_dates,
                                   "customer_email":touched_customer_emails,
                                   "channel":touch_channels})


## Write to disk

In [7]:
q2_table_purchases.to_csv('tables/sales.csv', index=False)
q3_table_marketing.to_csv('tables/marketing.csv', index=False)
customers_table.to_csv('tables/customers.csv', index=False)